# fase_3 - script_cimut Migration

This notebook handles migration of database from old DB to new DB for fase 3.

**Purpose**: Benerin database lama ke database baru untuk bagian CRM, Prospek, dan Operasional.

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import datetime
import random
import string
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## Ambil Data dari DB Lama

In [3]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS
# ---------------------------------------------------------
cursor_old.execute("SHOW TABLES")
tables_data = cursor_old.fetchall()
target_tables = [list(t.values())[0] for t in tables_data]
print(f"\n--- Ditemukan {len(target_tables)} tabel di Database Lama ---")

df_old = {}
for table in target_tables:
    try:
        query = f"SELECT * FROM `{table}`"
        df_old[table] = pd.read_sql(query, db_old)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_old[table])}")
    except Exception as e:
        print(f"Gagal load tabel {table}: {e}")

print("\n--- Proses load selesai. Semua data tersimpan di 'df_old' ---")


--- Ditemukan 108 tabel di Database Lama ---
Berhasil load tabel: absensi | Jumlah baris: 13444
Berhasil load tabel: absensi_note | Jumlah baris: 11
Berhasil load tabel: bidang | Jumlah baris: 4
Berhasil load tabel: bidangkategori | Jumlah baris: 12
Berhasil load tabel: bidanglink | Jumlah baris: 7
Berhasil load tabel: calon | Jumlah baris: 4
Berhasil load tabel: calon_detil | Jumlah baris: 61
Berhasil load tabel: calon_pertanyaan | Jumlah baris: 229
Berhasil load tabel: calon_pertanyaan_detil | Jumlah baris: 4305
Berhasil load tabel: catatan_kelas | Jumlah baris: 12797
Berhasil load tabel: catatan_kelas_tag | Jumlah baris: 999
Berhasil load tabel: catatan_mingguan | Jumlah baris: 0
Berhasil load tabel: catatan_siswa | Jumlah baris: 1502
Berhasil load tabel: catatan_siswa_follow_up | Jumlah baris: 22
Berhasil load tabel: catatanawal_admin | Jumlah baris: 64
Berhasil load tabel: catatanawal_datautama | Jumlah baris: 9
Berhasil load tabel: catatanawal_infolain | Jumlah baris: 64
Berhasi

## Ambil Data dari DB Baru (Struktur Target)

In [4]:
cursor_new.execute("SHOW TABLES")
tables_data_new = cursor_new.fetchall()
target_tables_new = [list(t.values())[0] for t in tables_data_new]
df_new = {}

for table in target_tables_new:
    try:
        query = f"SELECT * FROM `{table}`"
        df_new[table] = pd.read_sql(query, db_new)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_new[table])}")
    except:
        pass

Berhasil load tabel: absensi | Jumlah baris: 0
Berhasil load tabel: activity_log | Jumlah baris: 0
Berhasil load tabel: admin_sarpras | Jumlah baris: 1
Berhasil load tabel: bidang_kategori | Jumlah baris: 12
Berhasil load tabel: bidang_link | Jumlah baris: 7
Berhasil load tabel: busdev_bidang | Jumlah baris: 4
Berhasil load tabel: cache | Jumlah baris: 0
Berhasil load tabel: cache_locks | Jumlah baris: 0
Berhasil load tabel: calon_siswa | Jumlah baris: 165
Berhasil load tabel: calon_siswa_akademik | Jumlah baris: 338
Berhasil load tabel: calon_siswa_bayar | Jumlah baris: 338
Berhasil load tabel: calon_siswa_jadwal | Jumlah baris: 338
Berhasil load tabel: calon_siswa_kursus | Jumlah baris: 338
Berhasil load tabel: calon_siswa_ortu | Jumlah baris: 338
Berhasil load tabel: calon_siswa_proses | Jumlah baris: 338
Berhasil load tabel: calon_siswa_status_logs | Jumlah baris: 0
Berhasil load tabel: catatan_kelas | Jumlah baris: 25594
Berhasil load tabel: catatan_kelas_tag | Jumlah baris: 0
Ber

# Fixing kontak_prospek
Target: `kontak_prospek, calon_siswa, calon_siswa_akademik, calon_siswa_ortu, calon_siswa_bayar, calon_siswa_jadwal, calon_siswa_kursus, calon_siswa_proses, calon_siswa_status_logs, peminjaman, pengadaan, problem.`

In [5]:
display(df_new['kontak_prospek'])

,id_kontak_prospek,kode_kontak,nama_penanya,nomor_telepon,email,sumber_informasi,catatan_awal_fo,id_admin_fo,status_kontak,tanggal_kontak_pertama,tanggal_kontak_terakhir,created_at,updated_at
0,1,LKOQ92Z6KN,,083849241708,chyzryth@gmail.com,Lainnya,None,None,1,2025-09-17 11:34:19,NaT,2025-09-17 11:34:19,2026-05-26 01:13:34
1,2,JDHZJ5WSLO,,,ansaliapji@gmail.com,Teman/kerabat/saudara,None,None,1,2025-09-17 13:11:28,NaT,2025-09-17 13:11:28,2026-05-26 01:13:34
2,3,GB6LZ0P05P,,,virginialouisa1@gmail.com,Website,None,None,2,2025-09-20 06:43:28,NaT,2025-09-20 06:43:28,2026-05-26 01:13:34
3,4,NKJ93G8J8Q,,388424909,ula@beauty-fashion.cc,None,None,None,2,2025-10-01 05:44:23,NaT,2025-10-01 05:44:23,2026-05-26 01:13:34
4,5,L4QWC72JGF,,,agungdanuaga28@gmail.com,Website,None,None,3,2025-10-06 17:07:44,NaT,2025-10-06 17:07:44,2026-05-26 01:13:34
...,...,...,...,...,...,...,...,...,...,...,...,...,...
205,206,QO8DN8JMPH,,,cheryl2013sby@gmail.com,Teman/kerabat/saudara,None,None,on progress,2025-10-20 16:03:29,NaT,2025-10-24 15:23:53,2025-10-24 15:23:53
206,207,TC2D9LZWEW,,,asihasih180808@gmail.com,Teman/kerabat/saudara,None,None,on progress,2025-10-22 11:12:11,NaT,2025-10-24 15:24:48,2025-10-24 15:24:48
207,208,BDQE3OGKQK,,,None,None,None,None,follow up another time,2025-10-24 15:27:54,NaT,2025-10-24 15:27:54,2025-10-24 15:27:54
208,209,T46JLM6RUD,,,None,None,None,None,waiting for confirmation,2025-10-24 15:28:21,NaT,2025-10-24 15:28:21,2025-10-24 15:28:21


## Inspeksi Tabel Sumber (kontak_prospek)
Jalankan cell ini untuk melihat isi asli tabel-tabel di df_old yang berkaitan dengan kontak awal.

In [6]:
# source_tables = [
#     'catatanawal_datautama', 
#     'catatanawal_admin', 
#     'catatanawal_infolain', 
#     'catatanawal_tglpenting'
# ]

# print("--- INSPEKSI TABEL SUMBER ---")
# for t in source_tables:
#     if t in df_old:
#         print(f"\n>> Tabel: {t} ({len(df_old[t])} baris)")
#         print(f"Kolom: {df_old[t].columns.tolist()}")
#         display(df_old[t].head(3))
#     else:
#         print(f"\n>> Tabel {t} tidak ditemukan!")

## Tahap 1: Penggabungan (catatanawal_all)
Menggabungkan 4 tabel catatan awal menjadi satu tabel referensi lengkap.

In [7]:
def create_catatanawal_all():
    JOIN_KEY = 'idcatatanawal_admin'
    
    if 'catatanawal_admin' not in df_old:
        print("Error: catatanawal_admin tidak ditemukan!")
        return pd.DataFrame()

    # 1. Base Table: catatanawal_admin
    catatanawal_all = df_old['catatanawal_admin'].copy()
    
    # 2. Join dengan catatanawal_datautama
    if 'catatanawal_datautama' in df_old:
        catatanawal_all = pd.merge(catatanawal_all, df_old['catatanawal_datautama'], on=JOIN_KEY, how='left', suffixes=('', '_utama'))
    
    # 3. Join dengan catatanawal_infolain
    if 'catatanawal_infolain' in df_old:
        catatanawal_all = pd.merge(catatanawal_all, df_old['catatanawal_infolain'], on=JOIN_KEY, how='left', suffixes=('', '_info'))
        
    # 4. Join dengan catatanawal_tglpenting
    if 'catatanawal_tglpenting' in df_old:
        catatanawal_all = pd.merge(catatanawal_all, df_old['catatanawal_tglpenting'], on=JOIN_KEY, how='left', suffixes=('', '_tgl'))

    return catatanawal_all

# Buat tabel gabungan
catatanawal_all = create_catatanawal_all()

print(f"--- Tabel 'catatanawal_all' Berhasil Dibuat ---")
print(f"Total Baris: {len(catatanawal_all)}")
print(f"Total Kolom: {len(catatanawal_all.columns)}")
display(catatanawal_all.head())
print("\nKolom yang tersedia:")
print(catatanawal_all.columns.tolist())

--- Tabel 'catatanawal_all' Berhasil Dibuat ---
Total Baris: 64
Total Kolom: 54


,idcatatanawal_admin,nama,tlp,email,status,created_at,updated_at,idcatatanawal_datautama,pengontak_admin,nama_l,...,bank,wag_lv,pic,idcatatanawal_tglpenting,kontakA_date,wawancara_date,trial_date,tglbayar_date,tglmasuk_date,tglkeluar_date
0,M000000001,Ahmad 1,0812312344531,ahmad@gmail.com,canceled,2025-09-12 00:51:16,2025-09-12 00:51:16,NaN,NaN,NaN,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000000002,Daria Azmiya Jasmine,082231346758,puterihapsari.f@gmail.com,done,2025-09-15 13:33:49,2025-09-18 17:39:40,N000000001,Ibu Sari,Daria Azmiya Jasmine,...,Transfer Bank,,,O000000009,2025-09-15,None,2025-09-15,2025-09-17,2025-09-24,None
2,M000000003,Annisa Zahro Ramadhania,081336647476,dwirohm4@gmail.com,done,2025-09-16 11:13:41,2025-09-23 16:25:14,N000000002,Bu Dwi,Annisa Zahro Ramadhania,...,,,,O000000014,2025-09-16,2025-09-16,2025-09-16,None,2025-09-25,None
3,M000000004,Ghayda Syakira Hanania,081235160064,humaidah0208@gmail.com,done,2025-09-17 16:35:22,2025-10-03 09:47:53,N000000003,,Ghayda Syakira Hanania,...,,,,O000000012,2025-09-15,2025-09-15,2025-09-15,None,2025-09-25,None
4,M000000005,Achmad Naufal Albiruni,087765283592,melisnifuku@gmail.com,done,2025-09-17 16:49:20,2025-10-03 09:49:15,N000000004,Bu Lita,Achmad Naufal Albiruni,...,,,,O000000013,2025-09-17,2025-09-17,2025-09-17,None,2025-09-25,None



Kolom yang tersedia:
['idcatatanawal_admin', 'nama', 'tlp', 'email', 'status', 'created_at', 'updated_at', 'idcatatanawal_datautama', 'pengontak_admin', 'nama_l', 'nama_p', 'jenis_k', 'no_wa_ortu', 'no_wa_anak', 'email_utama', 'pilihan_program', 'jenis_program', 'level_1', 'level_2', 'tujuan_program', 'metode', 'info', 'referensi', 'sby_luarsby', 'kewarganegaraan', 'provinsi', 'kabupaten', 'kecamatan', 'kelurahan', 'alamat_lengkap', 'nama_instansi', 'kurikulum', 'pernah_les', 'kesulitan_pelajaran', 'idcatatanawal_infolain', 'jenis_test', 'keterangan', 'trial', 'hasil_test', 'wawancara', 'catatan_penting', 'diterima_dikelas', 'form_daftar', 'bulan_masuk', 'bank', 'wag_lv', 'pic', 'idcatatanawal_tglpenting', 'kontakA_date', 'wawancara_date', 'trial_date', 'tglbayar_date', 'tglmasuk_date', 'tglkeluar_date']


In [8]:
# 1. Hitung jumlah baris unik di tabel induk dan anak-anaknya
print("--- VERIFIKASI DATA MENTAH DB LAMA ---")
print("Total rows di catatanawal_admin (Induk):", len(df_old['catatanawal_admin']))
print("Total rows di catatanawal_datautama (Anak):", len(df_old['catatanawal_datautama']))
print("Total rows setelah kamu JOIN (Hasil fungsi):", len(catatanawal_all))

# 2. Cek apakah ada data anak yang hilang akibat left join
if len(catatanawal_all) < len(df_old['catatanawal_datautama']):
    print("\n⚠ PERINGATAN: Hasil gabungan lebih sedikit daripada data utama!")
    print("Ada data anak yang idcatatanawal_admin-nya tidak terdaftar di tabel induk.")
    print("Disarankan mengubah how='left' menjadi how='outer' atau membersihkan data NULL.")
else:
    print("\n✓ SEMPURNA! Semua data dari keempat tabel tercakup tanpa ada yang bocor.")

--- VERIFIKASI DATA MENTAH DB LAMA ---
Total rows di catatanawal_admin (Induk): 64
Total rows di catatanawal_datautama (Anak): 9
Total rows setelah kamu JOIN (Hasil fungsi): 64

✓ SEMPURNA! Semua data dari keempat tabel tercakup tanpa ada yang bocor.


In [9]:
def create_form_calon_all():
    JOIN_KEY_FORM = 'idcalon'
    
    if 'form_calon' not in df_old:
        print("Error: form_calon tidak ditemukan!")
        return pd.DataFrame()

    # 1. Base Table Form: form_calon
    form_calon_all = df_old['form_calon'].copy()
    
    # 2. Join dengan detail 1 sampai 4
    if 'form_calon_detil1' in df_old:
        form_calon_all = pd.merge(form_calon_all, df_old['form_calon_detil1'], on=JOIN_KEY_FORM, how='left', suffixes=('', '_dt1'))
        
    if 'form_calon_detil2' in df_old:
        form_calon_all = pd.merge(form_calon_all, df_old['form_calon_detil2'], on=JOIN_KEY_FORM, how='left', suffixes=('', '_dt2'))
        
    if 'form_calon_detil3' in df_old:
        form_calon_all = pd.merge(form_calon_all, df_old['form_calon_detil3'], on=JOIN_KEY_FORM, how='left', suffixes=('', '_dt3'))
        
    if 'form_calon_detil4' in df_old:
        form_calon_all = pd.merge(form_calon_all, df_old['form_calon_detil4'], on=JOIN_KEY_FORM, how='left', suffixes=('', '_dt4'))

    return form_calon_all

# Eksekusi penggabungan cluster form calon
form_calon_all = create_form_calon_all()
print(f"✓ Cluster 'form_calon_all' berhasil digabung. Total baris: {len(form_calon_all)}")
display(form_calon_all)
display(catatanawal_all)

✓ Cluster 'form_calon_all' berhasil digabung. Total baris: 184


,idcalon,fullName,email,nickName,phone1,phone2,schoolName,classLevel,gender,activities,...,nomor_invoice,bank,tanggal_pembayaran,bulan_masuk,idcalon_detil4,followUp1,followUp2,followUp3,keterangan_dt4,akun_lv
0,C00000017,Daria Azmiya Jasmine,puterihapsari.f@gmail.com,Daria,082231346758,,TK Al Maghfirah,TK B,Perempuan,None,...,,Mandiri,2025-09-17,September,Z00000017,None,None,None,,
1,C00000018,Annisa Zahro Ramadhania,dwirohm4@gmail.com,Annisa,081336647476,,Sd Khadijah wonorejo,Kelas 3,Perempuan,None,...,,Mandiri,2025-09-22,September,Z00000018,None,None,None,,
2,C00000019,Zulfa Bariatur Rahma,chyzryth@gmail.com,Zulfa,085707179656,083849241708,SMAN 17 Surabaya,X,None,None,...,,None,None,None,Z00000024,None,None,None,,
3,C00000020,Achmad Naufal Albiruni,melisnifuku@gmail.com,AlBI,087765283592,None,SD Khadijah Wonorejo Surabaya,SD kelas 5,Laki-laki,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,C00000021,Khansa Amalia Putri Aji,afadhilpa@gmail.com,Khansa,081554932188,,SMAN 17 Surabaya,XI,Perempuan,None,...,,Mandiri,2025-09-29,September,Z00000019,None,None,None,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179,C00000197,Antonius Miguel Kurniawan,adeodatus.kurniawan@gmail.com,Miguel,08170326911,None,SD Kartika Nasional Plus,SD Kelas 2,Laki-laki,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
180,C00000198,Tisha kayla janitra,nroskalindha18@gmail.com,Tisha,082244441630,None,Tk al fajar,Tk b,Perempuan,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181,C00000199,Clariza Arifianti,clarizarisa3@gmail.com,Clara,08977257033,None,None,None,Perempuan,Mahasiswa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
182,C00000200,Shaqila Anindra Dzakira,shaqilaanindra@gmail.com,Shaqila,085850209079,None,SDIT Ghilmani Surabaya,kelas 4,Perempuan,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,idcatatanawal_admin,nama,tlp,email,status,created_at,updated_at,idcatatanawal_datautama,pengontak_admin,nama_l,...,bank,wag_lv,pic,idcatatanawal_tglpenting,kontakA_date,wawancara_date,trial_date,tglbayar_date,tglmasuk_date,tglkeluar_date
0,M000000001,Ahmad 1,0812312344531,ahmad@gmail.com,canceled,2025-09-12 00:51:16,2025-09-12 00:51:16,NaN,NaN,NaN,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000000002,Daria Azmiya Jasmine,082231346758,puterihapsari.f@gmail.com,done,2025-09-15 13:33:49,2025-09-18 17:39:40,N000000001,Ibu Sari,Daria Azmiya Jasmine,...,Transfer Bank,,,O000000009,2025-09-15,None,2025-09-15,2025-09-17,2025-09-24,None
2,M000000003,Annisa Zahro Ramadhania,081336647476,dwirohm4@gmail.com,done,2025-09-16 11:13:41,2025-09-23 16:25:14,N000000002,Bu Dwi,Annisa Zahro Ramadhania,...,,,,O000000014,2025-09-16,2025-09-16,2025-09-16,None,2025-09-25,None
3,M000000004,Ghayda Syakira Hanania,081235160064,humaidah0208@gmail.com,done,2025-09-17 16:35:22,2025-10-03 09:47:53,N000000003,,Ghayda Syakira Hanania,...,,,,O000000012,2025-09-15,2025-09-15,2025-09-15,None,2025-09-25,None
4,M000000005,Achmad Naufal Albiruni,087765283592,melisnifuku@gmail.com,done,2025-09-17 16:49:20,2025-10-03 09:49:15,N000000004,Bu Lita,Achmad Naufal Albiruni,...,,,,O000000013,2025-09-17,2025-09-17,2025-09-17,None,2025-09-25,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,M000000060,Cheryl Kathleen Tan,0811537168,cheryl2013sby@gmail.com,on progress,2025-10-24 15:23:53,2025-10-24 15:23:53,NaN,NaN,NaN,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60,M000000061,Bimantara Sukma Arsa,08113322866,asihasih180808@gmail.com,on progress,2025-10-24 15:24:48,2025-10-24 15:24:48,NaN,NaN,NaN,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61,M000000062,Aca,082264644495,,follow up another time,2025-10-24 15:27:54,2025-10-24 15:27:54,NaN,NaN,NaN,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62,M000000063,Bu Hartik,0895368241226,,waiting for confirmation,2025-10-24 15:28:21,2025-10-24 15:28:21,NaN,NaN,NaN,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
import pandas as pd
import numpy as np

def create_mega_crm_all(df_catatan_all, df_form_all):
    print("=== MEMULAI PROSES MEGA OUTER JOIN (ANTI-DATA HILANG) ===")
    
    # 1. Copy data asli agar tidak merusak DataFrame awal
    catatan = df_catatan_all.copy()
    form = df_form_all.copy()
    
    # 2. Standarisasi Key untuk Join (Nama & Email dibuat huruf kecil & hapus spasi)
    catatan['join_nama'] = catatan['nama'].astype(str).str.strip().str.lower()
    catatan['join_email'] = catatan['email'].astype(str).str.strip().str.lower()
    
    form['join_nama'] = form['fullName'].astype(str).str.strip().str.lower()
    form['join_email'] = form['email'].astype(str).str.strip().str.lower()
    
    # 3. Eksekusi FULL OUTER JOIN menggunakan kombinasi Nama & Email
    mega_crm = pd.merge(
        catatan, 
        form, 
        on=['join_nama', 'join_email'], 
        how='outer', 
        suffixes=('_catatan', '_form')
    )
    
    # 4. Hapus kolom bantuan join agar data kembali bersih
    mega_crm.drop(columns=['join_nama', 'join_email'], errors='ignore', inplace=True)
    
    print(f"✓ Penggabungan selesai!")
    return mega_crm

# Eksekusi penggabungan takhta CRM kalian
mega_crm_all = create_mega_crm_all(catatanawal_all, form_calon_all)
display(mega_crm_all)

=== MEMULAI PROSES MEGA OUTER JOIN (ANTI-DATA HILANG) ===
✓ Penggabungan selesai!


,idcatatanawal_admin,nama,tlp,email_catatan,status_catatan,created_at_catatan,updated_at,idcatatanawal_datautama,pengontak_admin,nama_l,...,nomor_invoice,bank_form,tanggal_pembayaran,bulan_masuk_form,idcalon_detil4,followUp1,followUp2,followUp3,keterangan_dt4,akun_lv
0,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000000023,Abigail Caitlyn Wijaya,081351359474,fransisca_angilia@yahoo.co.id,done,2025-10-03 11:51:36,2025-10-16 10:44:02,NaN,NaN,NaN,...,,Mandiri,2025-09-25,September,Z00000013,None,None,None,,
2,M000000062,Aca,082264644495,,follow up another time,2025-10-24 15:27:54,2025-10-24 15:27:54,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M000000005,Achmad Naufal Albiruni,087765283592,melisnifuku@gmail.com,done,2025-09-17 16:49:20,2025-10-03 09:49:15,N000000004,Bu Lita,Achmad Naufal Albiruni,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
210,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
211,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
212,M000000007,Zulfa Bari'atur Rahma,083849241708,chyzryth@gmail.com,done,2025-09-17 17:32:16,2025-10-03 09:48:34,N000000006,Ibu Fitri,Zulfa Bari'atur Rahma,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
total_catatan = len(catatanawal_all)
total_form = len(form_calon_all)
total_gabungan = len(mega_crm_all)

print("--- LAPORAN AUDIT KEAMANAN DATA ---")
print(f"1. Total baris di cluster Catatan Awal  : {total_catatan}")
print(f"2. Total baris di cluster Form Calon    : {total_form}")
print(f"3. Total baris setelah Mega Outer Join  : {total_gabungan}")

# Hitung irisan data
hanya_di_catatan = mega_crm_all['idcatatanawal_admin'].notna() & mega_crm_all['idcalon'].isna()
hanya_di_form = mega_crm_all['idcatatanawal_admin'].isna() & mega_crm_all['idcalon'].notna()
ada_di_keduanya = mega_crm_all['idcatatanawal_admin'].notna() & mega_crm_all['idcalon'].notna()

print(f"\nAnalisis Isian Baris:")
print(f" • Siswa yang HANYA ada di inputan FO (Catatan Awal) : {hanya_di_catatan.sum()} baris")
print(f" • Siswa yang HANYA mendaftar lewat Web (Form Calon) : {hanya_di_form.sum()} baris")
print(f" • Siswa yang terdata di KEDUANYA (Sinkron)          : {ada_di_keduanya.sum()} baris")

# Rumus Matematika Outer Join: Total = A + B - Irisan
if total_gabungan == (hanya_di_catatan.sum() + hanya_di_form.sum() + ada_di_keduanya.sum()):
    print("\n✓ BERHASIL: Sistem mengonfirmasi 100% data aman. Tidak ada kebocoran atau data terbuang!")
else:
    print("\n⚠ PERINGATAN: Ada ketidaksesuaian kalkulasi pada baris gabungan.")

--- LAPORAN AUDIT KEAMANAN DATA ---
1. Total baris di cluster Catatan Awal  : 64
2. Total baris di cluster Form Calon    : 184
3. Total baris setelah Mega Outer Join  : 214

Analisis Isian Baris:
 • Siswa yang HANYA ada di inputan FO (Catatan Awal) : 30 baris
 • Siswa yang HANYA mendaftar lewat Web (Form Calon) : 146 baris
 • Siswa yang terdata di KEDUANYA (Sinkron)          : 38 baris

✓ BERHASIL: Sistem mengonfirmasi 100% data aman. Tidak ada kebocoran atau data terbuang!


In [12]:
import pandas as pd
import numpy as np

def consolidate_all_columns_final_v4(df_mega):
    print("=== MEMULAI KONSOLIDASI AKHIR SELURUH KOLOM (100% LENGKAP) ===")
    df_clean = pd.DataFrame()
    
    # 1. ID Historis Utama & ID Tabel Detail (Penting untuk Audit Internal)
    df_clean['id_prospek_lama'] = df_mega['idcatatanawal_admin']
    df_clean['id_form_lama'] = df_mega['idcalon']
    
    # 2. Identitas & Profil Siswa (LENGKAP)
    df_clean['nama_siswa'] = df_mega['nama_l'].combine_first(df_mega['fullName']).combine_first(df_mega['nama'])
    df_clean['nama_panggilan'] = df_mega['nama_p'].combine_first(df_mega['nickName'])
    df_clean['tempat_lahir_ortu'] = df_mega['tempat_lahir'] if 'tempat_lahir' in df_mega.columns else None
    df_clean['tanggal_lahir_ortu'] = df_mega['tanggal_lahir'] if 'tanggal_lahir' in df_mega.columns else None
    
    # Mapping Gender
    gender_map = {'l': 'Laki-laki', 'p': 'Perempuan', 'laki-laki': 'Laki-laki', 'perempuan': 'Perempuan'}
    raw_gender = df_mega['jenis_k'].combine_first(df_mega['gender']).astype(str).str.strip().str.lower()
    df_clean['jenis_kelamin'] = raw_gender.map(gender_map).replace({np.nan: None})
    
    # 3. Kontak & Hubungan Keluarga (LENGKAP)
    df_clean['email'] = df_mega['email_catatan'].combine_first(df_mega['email_form']).combine_first(df_mega['email_utama'])
    df_clean['wa_ortu'] = df_mega['no_wa_ortu'].combine_first(df_mega['phone2'])
    df_clean['wa_siswa'] = df_mega['no_wa_anak'].combine_first(df_mega['phone1'])
    df_clean['pengontak_admin'] = df_mega['pengontak_admin'] 
    df_clean['nama_ortu'] = df_mega['nama_ortu'] 
    df_clean['pekerjaan_ortu'] = df_mega['pekerjaan_ortu'] if 'pekerjaan_ortu' in df_mega.columns else None
    
    # 4. Program Pilihan & Leveling (LENGKAP)
    df_clean['pilihan_program'] = df_mega['pilihan_program'].combine_first(df_mega['program'])
    df_clean['jenis_program'] = df_mega['jenis_program']
    df_clean['level_1'] = df_mega['level_1'] if 'level_1' in df_mega.columns else None
    df_clean['level_2'] = df_mega['level_2'] if 'level_2' in df_mega.columns else None
    df_clean['wag_lv'] = df_mega['wag_lv'] if 'wag_lv' in df_mega.columns else None
    df_clean['akun_lv'] = df_mega['akun_lv'] if 'akun_lv' in df_mega.columns else None
    df_clean['tujuan_program'] = df_mega['tujuan_program'].combine_first(df_mega['purpose'])
    df_clean['placement'] = df_mega['placement'] if 'placement' in df_mega.columns else None
    
    # Kelas & Metode (Dipisah sesuai maumu)
    df_clean['metode_belajar_catatan'] = df_mega['metode'] 
    df_clean['class_options_form'] = df_mega['class_options'] 
    df_clean['diterima_dikelas'] = df_mega['diterima_dikelas'] if 'diterima_dikelas' in df_mega.columns else None
    
    # 5. Riwayat Sekolah & Pengalaman (LENGKAP)
    df_clean['nama_instansi_catatan'] = df_mega['nama_instansi'] 
    df_clean['school_name_form'] = df_mega['schoolName'] 
    df_clean['kelas_sekolah'] = df_mega['classLevel'] if 'classLevel' in df_mega.columns else None
    df_clean['kurikulum_sekolah'] = df_mega['kurikulum'].combine_first(df_mega['curriculum'])
    df_clean['pernah_les'] = df_mega['pernah_les'] if 'pernah_les' in df_mega.columns else None
    df_clean['kesulitan_pelajaran'] = df_mega['kesulitan_pelajaran'] if 'kesulitan_pelajaran' in df_mega.columns else None
    df_clean['exp_pengalaman'] = df_mega['exp'] if 'exp' in df_mega.columns else None
    
    # 6. Kuesioner Bakat / Aktivitas / Kebutuhan Hardware (LENGKAP)
    df_clean['activities'] = df_mega['activities'] if 'activities' in df_mega.columns else None
    df_clean['other_activities'] = df_mega['otherActivities'] if 'otherActivities' in df_mega.columns else None
    df_clean['office_app'] = df_mega['officeApp'] if 'officeApp' in df_mega.columns else None
    df_clean['editing_skill'] = df_mega['editing'] if 'editing' in df_mega.columns else None
    df_clean['custom_skill'] = df_mega['custom'] if 'custom' in df_mega.columns else None
    df_clean['computer_hardware'] = df_mega['computer'] if 'computer' in df_mega.columns else None
    df_clean['software_skill'] = df_mega['software'] if 'software' in df_mega.columns else None
    df_clean['gadget_used'] = df_mega['gadget'] if 'gadget' in df_mega.columns else None
    df_clean['student_hope'] = df_mega['hope'] if 'hope' in df_mega.columns else None
    df_clean['recommendation'] = df_mega['recom'] if 'recom' in df_mega.columns else None
    df_clean['other_detail'] = df_mega['otherDetail'] if 'otherDetail' in df_mega.columns else None
    df_clean['file_attachment'] = df_mega['file'] if 'file' in df_mega.columns else None
    
    # 7. Sesi Uji Coba, Wawancara, & Test (Form Detil 2 & Infolain)
    df_clean['jenis_test_catatan'] = df_mega['jenis_test_catatan'] if 'jenis_test_catatan' in df_mega.columns else None
    df_clean['jenis_test_form'] = df_mega['jenis_test'] if 'jenis_test' in df_mega.columns else None
    df_clean['hasil_test_catatan'] = df_mega['hasil_test'] if 'hasil_test' in df_mega.columns else None
    df_clean['laporan_test_form'] = df_mega['laporan_test'] if 'laporan_test' in df_mega.columns else None
    df_clean['diagnostic_test_form'] = df_mega['diagnostic'] if 'diagnostic' in df_mega.columns else None
    df_clean['wawancara_catatan'] = df_mega['wawancara'] if 'wawancara' in df_mega.columns else None
    df_clean['trial_catatan'] = df_mega['trial'] if 'trial' in df_mega.columns else None
    df_clean['trial_dimana_form'] = df_mega['trial_dimana'] if 'trial_dimana' in df_mega.columns else None
    df_clean['waktu_test1'] = df_mega['waktu_test1'] if 'waktu_test1' in df_mega.columns else None
    df_clean['waktu_test2'] = df_mega['waktu_test2'] if 'waktu_test2' in df_mega.columns else None
    
    # 8. Administrasi Kuitansi & Invoice (Form Detil 3)
    df_clean['nomor_invoice'] = df_mega['nomor_invoice'] if 'nomor_invoice' in df_mega.columns else None
    df_clean['bank_catatan'] = df_mega['bank_catatan'] if 'bank_catatan' in df_mega.columns else None
    df_clean['bank_form'] = df_mega['bank_form'] if 'bank_form' in df_mega.columns else None
    df_clean['bulan_masuk_catatan'] = df_mega['bulan_masuk_catatan'] if 'bulan_masuk_catatan' in df_mega.columns else None
    df_clean['bulan_masuk_form'] = df_mega['bulan_masuk_form'] if 'bulan_masuk_form' in df_mega.columns else None
    df_clean['status_siswa_form'] = df_mega['status_siswa'] if 'status_siswa' in df_mega.columns else None
    df_clean['lokasi_pendaftaran'] = df_mega['lokasi'] if 'lokasi' in df_mega.columns else None
    
    # 9. Follow Up Logs & Catatan FO (Form Detil 4 & Infolain)
    df_clean['follow_up1'] = df_mega['followUp1'] if 'followUp1' in df_mega.columns else None
    df_clean['follow_up2'] = df_mega['followUp2'] if 'followUp2' in df_mega.columns else None
    df_clean['follow_up3'] = df_mega['followUp3'] if 'followUp3' in df_mega.columns else None
    df_clean['catatan_penting_catatan'] = df_mega['catatan_penting'] if 'catatan_penting' in df_mega.columns else None
    df_clean['keterangan_catatan'] = df_mega['keterangan_catatan'] if 'keterangan_catatan' in df_mega.columns else None
    df_clean['keterangan_form'] = df_mega['keterangan_form'] if 'keterangan_form' in df_mega.columns else None
    df_clean['catatan_admin_form'] = df_mega['catatanadmin'] if 'catatanadmin' in df_mega.columns else None
    df_clean['status_terakhir'] = df_mega['status_catatan'].combine_first(df_mega['status_form'])
    df_clean['pic_admin'] = df_mega['pic'] if 'pic' in df_mega.columns else None
    df_clean['id_pend_kursus'] = df_mega['idpendkursus'] if 'idpendkursus' in df_mega.columns else None
    
    # 10. Detail Geografis Wilayah Lama
    df_clean['provinsi_id_lama'] = df_mega['provinsi_catatan'].combine_first(df_mega['provinsi_form'])
    df_clean['kabupaten_id_lama'] = df_mega['kabupaten_catatan'].combine_first(df_mega['kabupaten_form'])
    df_clean['kecamatan_id_lama'] = df_mega['kecamatan'] if 'kecamatan' in df_mega.columns else None
    df_clean['kelurahan_id_lama'] = df_mega['kelurahan'] if 'kelurahan' in df_mega.columns else None
    df_clean['alamat_lengkap'] = df_mega['alamat_lengkap'] if 'alamat_lengkap' in df_mega.columns else None
    df_clean['sby_luarsby'] = df_mega['sby_luarsby'] if 'sby_luarsby' in df_mega.columns else None
    df_clean['kewarganegaraan'] = df_mega['kewarganegaraan_catatan'].combine_first(df_mega['kewarganegaraan_form'])
    
    # 11. Log Semua Tanggal Penting Historis
    df_clean['tanggal_kontak_awal'] = df_mega['kontakA_date'].combine_first(df_mega['created_at_form']).combine_first(df_mega['created_at_catatan'])
    df_clean['tanggal_wawancara'] = df_mega['wawancara_date']
    df_clean['tanggal_trial'] = df_mega['trial_date'].combine_first(df_mega['tgl_trial'])
    df_clean['tanggal_pembayaran'] = df_mega['tglbayar_date'].combine_first(df_mega['tanggal_pembayaran'])
    df_clean['tanggal_masuk'] = df_mega['tglmasuk_date'] if 'tglmasuk_date' in df_mega.columns else None
    df_clean['tanggal_keluar'] = df_mega['tglkeluar_date'] if 'tglkeluar_date' in df_mega.columns else None
    df_clean['sumber_informasi'] = df_mega['info_catatan'].combine_first(df_mega['info_form'])
    df_clean['referensi'] = df_mega['referensi'] if 'referensi' in df_mega.columns else None
    
    # 12. Metadata Log Dasar Aplikasi Lama
    df_clean['created_at_asli'] = df_mega['created_at_catatan'].combine_first(df_mega['created_at_form'])
    df_clean['updated_at_asli'] = df_mega['updated_at']

    # PEMBERSIHAN MUTLAK: Ubah string 'nan' bawaan Pandas menjadi tipe None objek asli Python
    for col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(lambda x: None if pd.isna(x) or str(x).strip().lower() == 'nan' or str(x).strip() == '' else x)
        
    print(f"✓ SELESAI! Seluruh data dari tabel detail berhasil disatukan ke dalam {len(df_clean.columns)} kolom.")
    return df_clean

# Jalankan konsolidasi raksasa 100% lengkap
df_crm_consolidated = consolidate_all_columns_final_v4(mega_crm_all)
display(mega_crm_all)
display(df_crm_consolidated)

=== MEMULAI KONSOLIDASI AKHIR SELURUH KOLOM (100% LENGKAP) ===
✓ SELESAI! Seluruh data dari tabel detail berhasil disatukan ke dalam 87 kolom.


,idcatatanawal_admin,nama,tlp,email_catatan,status_catatan,created_at_catatan,updated_at,idcatatanawal_datautama,pengontak_admin,nama_l,...,nomor_invoice,bank_form,tanggal_pembayaran,bulan_masuk_form,idcalon_detil4,followUp1,followUp2,followUp3,keterangan_dt4,akun_lv
0,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000000023,Abigail Caitlyn Wijaya,081351359474,fransisca_angilia@yahoo.co.id,done,2025-10-03 11:51:36,2025-10-16 10:44:02,NaN,NaN,NaN,...,,Mandiri,2025-09-25,September,Z00000013,None,None,None,,
2,M000000062,Aca,082264644495,,follow up another time,2025-10-24 15:27:54,2025-10-24 15:27:54,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M000000005,Achmad Naufal Albiruni,087765283592,melisnifuku@gmail.com,done,2025-09-17 16:49:20,2025-10-03 09:49:15,N000000004,Bu Lita,Achmad Naufal Albiruni,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
210,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
211,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
212,M000000007,Zulfa Bari'atur Rahma,083849241708,chyzryth@gmail.com,done,2025-09-17 17:32:16,2025-10-03 09:48:34,N000000006,Ibu Fitri,Zulfa Bari'atur Rahma,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,id_prospek_lama,id_form_lama,nama_siswa,nama_panggilan,tempat_lahir_ortu,tanggal_lahir_ortu,jenis_kelamin,email,wa_ortu,wa_siswa,...,tanggal_kontak_awal,tanggal_wawancara,tanggal_trial,tanggal_pembayaran,tanggal_masuk,tanggal_keluar,sumber_informasi,referensi,created_at_asli,updated_at_asli
0,None,C00000131,42234234234432,42234234234432,None,None,Laki-laki,42234234234432@gmail.com,None,081334223423,...,2026-01-08 16:50:38,None,None,None,None,None,Teman/kerabat/saudara,None,2026-01-08 16:50:38,NaT
1,M000000023,C00000036,Abigail Caitlyn Wijaya,Caitlyn,None,None,Perempuan,fransisca_angilia@yahoo.co.id,None,081351359474,...,2025-09-24 17:40:39,None,2025-09-24,2025-09-25,None,None,Website,None,2025-10-03 11:51:36,2025-10-16 10:44:02
2,M000000062,None,Aca,None,None,None,None,None,None,None,...,2025-10-24 15:27:54,None,None,None,None,None,None,None,2025-10-24 15:27:54,2025-10-24 15:27:54
3,M000000005,C00000020,Achmad Naufal Albiruni,Albi,None,None,Laki-laki,melisnifuku@gmail.com,087765283592,None,...,2025-09-17,2025-09-17,2025-09-17,None,2025-09-25,None,None,None,2025-09-17 16:49:20,2025-10-03 09:49:15
4,None,C00000169,Achmad Ryan,ryan,None,None,Laki-laki,achmadryanivansyah@gmail.com,None,6281231429361,...,2026-02-17 01:21:36,None,None,None,None,None,Tiktok,None,2026-02-17 01:21:36,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209,None,C00000182,zein achmat alghifari,zein,None,None,Laki-laki,zeinstyles74@gmail.con,None,0881026132208,...,2026-03-08 04:57:09,None,None,None,None,None,Tiktok,None,2026-03-08 04:57:09,NaT
210,None,C00000158,zila,zila,None,None,Perempuan,almasazzilah02@gmail.com,None,085257171654,...,2026-02-04 15:05:56,None,None,None,None,None,Website,None,2026-02-04 15:05:56,NaT
211,None,C00000146,Zufar Tazakka Hanif,Zakka,None,None,Laki-laki,zufarzakka@gmail.com,None,081358401440,...,2026-01-20 10:45:40,None,None,None,None,None,Teman/kerabat/saudara,None,2026-01-20 10:45:40,NaT
212,M000000007,None,Zulfa Bari'atur Rahma,Zulfa,None,None,Perempuan,chyzryth@gmail.com,083849241708,085707179656,...,2025-09-16,None,None,None,None,None,None,None,2025-09-17 17:32:16,2025-10-03 09:48:34


In [13]:
mega_crm_all['hasil_test'].value_counts(dropna=False)

hasil_test
NaN    146
        68
Name: count, dtype: int64

In [14]:
def check_and_remove_duplicates(df_clean):
    print("\n=== MEMULAI PENGECEKAN DATA DUPLIKAT ===")
    
    # Buat kolom bayangan untuk pengecekan tanpa merusak data asli
    check_nama = df_clean['nama_siswa'].astype(str).str.strip().str.lower()
    check_email = df_clean['email'].astype(str).str.strip().str.lower()
    
    # 1. Cari baris mana saja yang duplikat
    duplicate_mask = df_clean.duplicated(subset=['nama_siswa', 'email'], keep=False) & df_clean['nama_siswa'].notna()
    df_duplicates = df_clean[duplicate_mask].sort_values(by='nama_siswa')
    
    total_duplikat = df_clean.duplicated(subset=['nama_siswa', 'email'], keep='first').sum()
    print(f"• Ditemukan sekitar {total_duplikat} baris data yang terindikasi DUPLIKAT.")
    
    if total_duplikat > 0:
        print("\nContoh data duplikat yang ditemukan:")
        display(df_duplicates[['id_prospek_lama', 'id_form_lama', 'nama_siswa', 'email']].head(6))
        
        # 2. Hapus duplikat, pertahankan baris pertama yang muncul
        df_final = df_clean.drop_duplicates(subset=['nama_siswa', 'email'], keep='first')
        print(f"\n✓ Sukses membuang data duplikat! Total data menyusut dari {len(df_clean)} menjadi {len(df_final)} baris.")
        return df_final
    else:
        print("✓ Sempurna! Tidak ada data duplikat berdasarkan Nama + Email.")
        return df_clean

# Eksekusi Pembersihan Duplikat Akhir
df_crm_final_clean = check_and_remove_duplicates(df_crm_consolidated)


=== MEMULAI PENGECEKAN DATA DUPLIKAT ===
• Ditemukan sekitar 13 baris data yang terindikasi DUPLIKAT.

Contoh data duplikat yang ditemukan:


,id_prospek_lama,id_form_lama,nama_siswa,email
6,None,C00000118,Adara Bellvania Fredella Firmansyah,nidiafetry20@gmail.com
7,None,C00000128,Adara Bellvania Fredella Firmansyah,nidiafetry20@gmail.com
62,M000000019,C00000043,Diana Mandasari,dianamandrasari@gmail.com
63,M000000019,C00000044,Diana Mandasari,dianamandrasari@gmail.com
64,M000000019,C00000045,Diana Mandasari,dianamandrasari@gmail.com
102,None,C00000124,Keana Belva,rsielviana12@gmail.com



✓ Sukses membuang data duplikat! Total data menyusut dari 214 menjadi 201 baris.


In [15]:
import pandas as pd

def intip_data_duplikat(df_consolidated):
    print("=== TAMPILAN DATA MURID YANG DUPLIKAT ===")
    
    # 1. Buat kolom bantuan matching (lowercase + tanpa spasi)
    df_temp = df_consolidated.copy()
    df_temp['match_nama'] = df_temp['nama_siswa'].astype(str).str.strip().str.lower()
    df_temp['match_email'] = df_temp['email'].astype(str).str.strip().str.lower()
    
    # 2. Filter baris yang benar-benar kembar (Nama & Email sama, dan tidak kosong)
    duplicate_mask = df_temp.duplicated(subset=['match_nama', 'match_email'], keep=False) & df_temp['nama_siswa'].notna()
    
    # 3. Urutkan berdasarkan Nama agar data yang kembar berbaris rapi atas-bawah
    df_duplicates = df_temp[duplicate_mask].sort_values(by=['match_nama'])
    
    total_baris_kembar = len(df_duplicates)
    print(f"• Ditemukan total {total_baris_kembar} baris yang saling menduplikasi.\n")
    
    if total_baris_kembar > 0:
        # Tampilkan kolom index asli bawaan pandas, ID lama, nama, email, dan tanggalnya
        # Kamu bisa tambah kolom lain di dalam list jika ingin melihat isian lainnya
        kolom_intip = [
            'id_prospek_lama', 'id_form_lama', 'nama_siswa', 'email', 
            'pengontak_admin', 'nama_ortu', 'pilihan_program', 'tanggal_kontak_awal'
        ]
        
        # Tampilkan DataFrame duplikat lengkap dengan nomor INDEX aslinya di sebelah kiri
        display(df_duplicates)
    else:
        print("✓ Bersih! Tidak ada data duplikat sama sekali.")
        
    return df_temp

# Jalankan pengintipan data duplikat
df_bantuan_duplikat = intip_data_duplikat(df_crm_consolidated)

=== TAMPILAN DATA MURID YANG DUPLIKAT ===
• Ditemukan total 22 baris yang saling menduplikasi.



,id_prospek_lama,id_form_lama,nama_siswa,nama_panggilan,tempat_lahir_ortu,tanggal_lahir_ortu,jenis_kelamin,email,wa_ortu,wa_siswa,...,tanggal_trial,tanggal_pembayaran,tanggal_masuk,tanggal_keluar,sumber_informasi,referensi,created_at_asli,updated_at_asli,match_nama,match_email
6,None,C00000118,Adara Bellvania Fredella Firmansyah,Bellva,None,None,Perempuan,nidiafetry20@gmail.com,None,081241177889,...,None,None,None,None,Instagram,None,2026-01-05 18:34:07,NaT,adara bellvania fredella firmansyah,nidiafetry20@gmail.com
7,None,C00000128,Adara Bellvania Fredella Firmansyah,Bellva,None,None,Perempuan,nidiafetry20@gmail.com,None,081241177889,...,2026-01-07,2026-01-07,None,None,Instagram,None,2026-01-07 18:45:12,NaT,adara bellvania fredella firmansyah,nidiafetry20@gmail.com
62,M000000019,C00000043,Diana Mandasari,Diana,None,None,Perempuan,dianamandrasari@gmail.com,None,0896-1347-1313,...,None,None,None,None,Lainnya,None,2025-10-02 17:54:35,2025-10-02 17:54:35,diana mandasari,dianamandrasari@gmail.com
63,M000000019,C00000044,Diana Mandasari,Diana,None,None,Perempuan,dianamandrasari@gmail.com,None,0896-1347-1313,...,None,None,None,None,Lainnya,None,2025-10-02 17:54:35,2025-10-02 17:54:35,diana mandasari,dianamandrasari@gmail.com
64,M000000019,C00000045,Diana Mandasari,Diana,None,None,Perempuan,dianamandrasari@gmail.com,None,0896-1347-1313,...,None,None,None,None,Lainnya,None,2025-10-02 17:54:35,2025-10-02 17:54:35,diana mandasari,dianamandrasari@gmail.com
102,None,C00000124,Keana Belva,Nana,None,None,Perempuan,rsielviana12@gmail.com,None,085815765936,...,2026-01-07,2026-01-19,None,None,Teman/kerabat/saudara,None,2026-01-07 14:46:55,NaT,keana belva,rsielviana12@gmail.com
103,None,C00000125,Keana Belva,Nana,None,None,Perempuan,rsielviana12@gmail.com,None,085815765936,...,None,None,None,None,Teman/kerabat/saudara,None,2026-01-07 14:48:34,NaT,keana belva,rsielviana12@gmail.com
125,None,C00000112,MAGHROBI SUWIGNYO,ROBI,None,None,Laki-laki,maghrobisuwignyo6@gmail.com,None,08976405301,...,None,None,None,None,Tiktok,None,2026-01-03 13:25:36,NaT,maghrobi suwignyo,maghrobisuwignyo6@gmail.com
126,None,C00000113,MAGHROBI SUWIGNYO,ROBI,None,None,Laki-laki,maghrobisuwignyo6@gmail.com,None,08976405301,...,None,None,None,None,Tiktok,None,2026-01-03 13:29:39,NaT,maghrobi suwignyo,maghrobisuwignyo6@gmail.com
135,M000000025,C00000040,Mikhayla Dayu Mehrunisa,Mikha,None,None,Perempuan,novi.arch8@gmail.com,None,081231069008,...,2025-10-06,2025-10-07,None,None,Teman/kerabat/saudara,None,2025-10-03 12:36:06,2025-10-16 11:32:36,mikhayla dayu mehrunisa,novi.arch8@gmail.com


In [16]:
# === MASUKKAN PILIHANMU DI SINI ===
# Ganti angka di dalam kurung siku dengan nomor index yang ingin kamu hapus hasil dari intipan di atas
index_yang_mau_didrop = [] # <--- Contoh, silakan ganti dengan nomor index aslimu

print(f"Memproses penghapusan {len(index_yang_mau_didrop)} baris pilihan Cimut...")

# Eksekusi penghapusan berdasarkan nomor index yang kamu pilih
df_crm_final_clean = df_crm_consolidated.drop(index=index_yang_mau_didrop)

print(f"✓ Sukses! Baris pilihan berhasil dihapus.")
print(f"Total data sekarang menjadi: {len(df_crm_final_clean)} baris.")

Memproses penghapusan 0 baris pilihan Cimut...
✓ Sukses! Baris pilihan berhasil dihapus.
Total data sekarang menjadi: 214 baris.


# daerah

In [17]:
import pandas as pd
import numpy as np

def pre_mapping_cleaner_final(df_siswa, df_kab_lama):
    print("==================================================")
    print("  PRE-MAPPING CLEANER (VERSI HARD-FIX MANUAL)    ")
    print("==================================================")
    
    # 1. Copy data agar aman tidak merusak DataFrame asli
    df_siswa_clean = df_siswa.copy()
    df_kab_clean = df_kab_lama.copy()
    
    if 'name' in df_kab_clean.columns:
        # --- FIXING KASUS SURABAYA ---
        mask_sby = df_kab_clean['name'].astype(str).str.strip().str.upper() == 'SURABAYA'
        df_kab_clean.loc[mask_sby, 'name'] = 'KOTA SURABAYA'
        print(f"✓ Sukses memperbaiki {mask_sby.sum()} baris master 'SURABAYA' -> 'KOTA SURABAYA' di DB Lama.")
        
        # --- FIXING KASUS JAKARTA MANUAL (TEMBAK LANGSUNG SESUAI MAU CIMUT) ---
        # Kamus jembatan manual khusus untuk Jakarta agar teksnya pas pasrah 100% klop
        kamus_jakarta_manual = {
            'KOTA ADM. JAKARTA PUSAT': 'Kota Administrasi Jakarta Pusat',
            'KOTA ADM. JAKARTA UTARA': 'Kota Administrasi Jakarta Utara',
            'KOTA ADM. JAKARTA BARAT': 'Kota Administrasi Jakarta Barat',
            'KOTA ADM. JAKARTA SELATAN': 'Kota Administrasi Jakarta Selatan',
            'KOTA ADM. JAKARTA TIMUR': 'Kota Administrasi Jakarta Timur'
        }
        
        total_jkt_fixed = 0
        for idx, row in df_kab_clean.iterrows():
            nama_asal_raw = str(row['name']).strip()
            
            # Jika namanya persis ada di kamus manual, langsung ganti nilainya mentah-mentah
            if nama_asal_raw in kamus_jakarta_manual:
                df_kab_clean.at[idx, 'name'] = kamus_jakarta_manual[nama_asal_raw]
                total_jkt_fixed += 1
                
        print(f"✓ Sukses Hard-Fix {total_jkt_fixed} baris DKI Jakarta ke format 'Kota Administrasi' di DB Lama.")

    # 3. FIXING VALUE O (NOL)
    kolom_wilayah = ['provinsi_id_lama', 'kabupaten_id_lama', 'kecamatan_id_lama', 'kelurahan_id_lama']
    for col in kolom_wilayah:
        if col in df_siswa_clean.columns:
            kondisi_nol = (df_siswa_clean[col] == 0) | (df_siswa_clean[col] == 0.0) | (df_siswa_clean[col].astype(str).str.strip() == '0')
            total_nol = kondisi_nol.sum()
            df_siswa_clean.loc[kondisi_nol, col] = None
            print(f"✓ Kolom '{col}': Berhasil membersihkan {total_nol} data bernilai '0' menjadi NULL (None).")
            
    print("\n✓ Tahap Pre-processing Manual Jakarta Selesai! Data steril.")
    return df_siswa_clean, df_kab_clean

# === JALANKAN DI SEL NOTEBOOK-MU ===
df_crm_final_clean, df_old['kabupaten'] = pre_mapping_cleaner_final(df_crm_final_clean, df_old['kabupaten'])

  PRE-MAPPING CLEANER (VERSI HARD-FIX MANUAL)    
✓ Sukses memperbaiki 1 baris master 'SURABAYA' -> 'KOTA SURABAYA' di DB Lama.
✓ Sukses Hard-Fix 5 baris DKI Jakarta ke format 'Kota Administrasi' di DB Lama.
✓ Kolom 'provinsi_id_lama': Berhasil membersihkan 9 data bernilai '0' menjadi NULL (None).
✓ Kolom 'kabupaten_id_lama': Berhasil membersihkan 10 data bernilai '0' menjadi NULL (None).
✓ Kolom 'kecamatan_id_lama': Berhasil membersihkan 9 data bernilai '0' menjadi NULL (None).
✓ Kolom 'kelurahan_id_lama': Berhasil membersihkan 9 data bernilai '0' menjadi NULL (None).

✓ Tahap Pre-processing Manual Jakarta Selesai! Data steril.


In [18]:
import pandas as pd
import numpy as np

def clean_region_general(name):
    """Pembersihan umum untuk Provinsi, Kecamatan, dan Kelurahan."""
    if pd.isna(name) or not name: 
        return ""
    name = str(name).strip().lower()
    name = name.replace("provinsi", "").replace("kec.", "").replace("kel.", "")
    return " ".join(name.split())

def clean_kabupaten_name(name):
    """Pembersihan khusus Kabupaten/Kota sesuai spesifikasi Cimut."""
    if pd.isna(name) or not name: 
        return ""
    name = str(name).strip().lower()
    
    if "kota" in name:
        # Pertahankan kata 'kota ', buang embel-embel kab jika tertulis ganda
        name = "kota " + name.replace("kab.", "").replace("kabupaten", "").replace("kota", "").strip()
    else:
        # Jika murni kabupaten, hapus penanda jenisnya, sisakan nama saja
        name = name.replace("kab.", "").replace("kabupaten", "").strip()
        
    return " ".join(name.split())

def repair_and_normalize_id(val):
    """Memperbaiki kode ID desimal float (.0) akibat pembacaan file excel/csv."""
    if pd.isna(val) or str(val).strip() == "" or str(val).strip().lower() == "nan":
        return None
    val_str = str(val).strip()
    if val_str.endswith('.0'):
        val_str = val_str[:-2]
    return val_str

print("==================================================")
print("      MEMBANGUN REVISI KAMUS JEMBATAN REGION      ")
print("==================================================")

# 1. Jembatan Provinsi
df_old_prov = df_old['provinsi'].copy()
df_new_prov = df_new['provinsi'].copy()
df_old_prov['idprovinsi'] = df_old_prov['idprovinsi'].apply(repair_and_normalize_id)
df_old_prov['clean'] = df_old_prov['nama'].apply(clean_region_general)
df_new_prov['clean'] = df_new_prov['nama_provinsi'].apply(clean_region_general)

map_name_to_id_prov = dict(zip(df_new_prov['clean'], df_new_prov['id_provinsi']))
map_provinsi_id = {str(row['idprovinsi']).strip(): map_name_to_id_prov.get(row['clean'], None) for _, row in df_old_prov.iterrows() if row['idprovinsi'] is not None}


# 2. Jembatan Kabupaten (Menerapkan Fungsi Spesifik Anti-Tabrakan Kediri)
df_old_kab = df_old['kabupaten'].copy()
df_new_kab = df_new['kabupaten'].copy()
df_old_kab['idkabupaten'] = df_old_kab['idkabupaten'].apply(repair_and_normalize_id)
df_old_kab['idprovinsi'] = df_old_kab['idprovinsi'].apply(repair_and_normalize_id)

df_old_kab['clean'] = df_old_kab['name'].apply(clean_kabupaten_name)
df_new_kab['clean'] = df_new_kab['nama_kabupaten'].apply(clean_kabupaten_name)

map_name_to_id_kab = dict(zip(df_new_kab['clean'], df_new_kab['id_kabupaten']))
map_kabupaten_id = {str(row['idkabupaten']).strip(): map_name_to_id_kab.get(row['clean'], None) for _, row in df_old_kab.iterrows() if row['idkabupaten'] is not None}


# 3. Jembatan Kecamatan (Mengikat ke ID Kabupaten Baru agar Aman dari Nama Kembar)
df_old_kec = df_old['kecamatan'].copy()
df_new_kec = df_new['kecamatan'].copy()
df_old_kec['idkecamatan'] = df_old_kec['idkecamatan'].apply(repair_and_normalize_id)
df_old_kec['idkabupaten'] = df_old_kec['idkabupaten'].apply(repair_and_normalize_id)

df_old_kec['clean'] = df_old_kec['nama'].apply(clean_region_general)
df_new_kec['clean'] = df_new_kec['nama_kecamatan'].apply(clean_region_general)
df_new_kec['match_key'] = df_new_kec['id_kabupaten'].astype(str) + "_" + df_new_kec['clean']
key_to_id_kec_baru = dict(zip(df_new_kec['match_key'], df_new_kec['id_kecamatan']))

map_kecamatan_id = {}
for _, row in df_old_kec.iterrows():
    id_kab_lama = str(row['idkabupaten']).strip()
    id_kab_baru = map_kabupaten_id.get(id_kab_lama, None)
    if id_kab_baru:
        match_key = f"{int(id_kab_baru)}_{row['clean']}"
        map_kecamatan_id[str(row['idkecamatan']).strip()] = key_to_id_kec_baru.get(match_key, None)
    else:
        map_kecamatan_id[str(row['idkecamatan']).strip()] = None


# 4. Jembatan Kelurahan (Mengikat ke ID Kecamatan Baru agar Aman dari Nama Kembar)
df_old_kel = df_old['kelurahan'].copy()
df_new_kel = df_new['kelurahan'].copy()
df_old_kel['idkelurahan'] = df_old_kel['idkelurahan'].apply(repair_and_normalize_id)
df_old_kel['idkecamatan'] = df_old_kel['idkecamatan'].apply(repair_and_normalize_id)

df_old_kel['clean'] = df_old_kel['nama'].apply(clean_region_general)
df_new_kel['clean'] = df_new_kel['nama_kelurahan'].apply(clean_region_general)
df_new_kel['match_key'] = df_new_kel['id_kecamatan'].astype(str) + "_" + df_new_kel['clean']
key_to_id_kel_baru = dict(zip(df_new_kel['match_key'], df_new_kel['id_kelurahan']))

map_kelurahan_id = {}
for _, row in df_old_kel.iterrows():
    id_kec_lama = str(row['idkecamatan']).strip()
    id_kec_baru = map_kecamatan_id.get(id_kec_lama, None)
    if id_kec_baru:
        match_key = f"{int(id_kec_baru)}_{row['clean']}"
        map_kelurahan_id[str(row['idkelurahan']).strip()] = key_to_id_kel_baru.get(match_key, None)
    else:
        map_kelurahan_id[str(row['idkelurahan']).strip()] = None

print("\n✓ Hasil Audit Kunci Jembatan:")
print(f" • Pemetaan Provinsi Terkunci  : {len([k for k,v in map_provinsi_id.items() if v is not None])} daerah.")
print(f" • Pemetaan Kabupaten Terkunci : {len([k for k,v in map_kabupaten_id.items() if v is not None])} daerah.")
print(f" • Pemetaan Kecamatan Terkunci : {len([k for k,v in map_kecamatan_id.items() if v is not None])} daerah.")
print(f" • Pemetaan Kelurahan Terkunci : {len([k for k,v in map_kelurahan_id.items() if v is not None])} daerah.")

      MEMBANGUN REVISI KAMUS JEMBATAN REGION      

✓ Hasil Audit Kunci Jembatan:
 • Pemetaan Provinsi Terkunci  : 34 daerah.
 • Pemetaan Kabupaten Terkunci : 513 daerah.
 • Pemetaan Kecamatan Terkunci : 7160 daerah.
 • Pemetaan Kelurahan Terkunci : 81371 daerah.


In [19]:
def konversi_wilayah_siswa_final_v5(df_clean_siswa):
    print("\n=== MENULAI EKSEKUSI KONVERSI WILAYAH SISWA ===")
    df_res = df_clean_siswa.copy()
    
    # Perbaiki tipe desimal float jika ada di records siswa
    df_res['provinsi_id_lama'] = df_res['provinsi_id_lama'].apply(repair_and_normalize_id)
    df_res['kabupaten_id_lama'] = df_res['kabupaten_id_lama'].apply(repair_and_normalize_id)
    df_res['kecamatan_id_lama'] = df_res['kecamatan'].apply(repair_and_normalize_id) if 'kecamatan' in df_res.columns else None
    df_res['kelurahan_id_lama'] = df_res['kelurahan'].apply(repair_and_normalize_id) if 'kelurahan' in df_res.columns else None

    # Tembak ID Baru dari Kamus Jembatan Hirarki
    df_res['id_provinsi_baru'] = df_res['provinsi_id_lama'].apply(lambda x: map_provinsi_id.get(str(x).strip(), None) if x else None)
    df_res['id_kabupaten_baru'] = df_res['kabupaten_id_lama'].apply(lambda x: map_kabupaten_id.get(str(x).strip(), None) if x else None)
    df_res['id_kecamatan_baru'] = df_res['kecamatan_id_lama'].apply(lambda x: map_kecamatan_id.get(str(x).strip(), None) if x else None)
    df_res['id_kelurahan_baru'] = df_res['kelurahan_id_lama'].apply(lambda x: map_kelurahan_id.get(str(x).strip(), None) if x else None)
    
    print("✓ Sukses! Seluruh data wilayah siswa berhasil dikonversi sempurna.")
    return df_res

# Jalankan pada DataFrame final-mu
df_crm_final_with_region = konversi_wilayah_siswa_final_v5(df_crm_final_clean)


=== MENULAI EKSEKUSI KONVERSI WILAYAH SISWA ===
✓ Sukses! Seluruh data wilayah siswa berhasil dikonversi sempurna.


In [20]:
import pandas as pd

def display_sebelum_dan_sesudah_wilayah(df_hasil_konversi):
    print("=========================================================================")
    print("      AUDIT DISPLAY: PERBANDINGAN DATA WILAYAH (SEBELUM VS SESUDAH)      ")
    print("=========================================================================\n")
    
    # 1. Pilih kolom-kolom perbandingan agar ringkas saat di-display
    kolom_audit = [
        'nama_siswa',
        # --- SEBELUM ---
        'provinsi_id_lama', 
        'kabupaten_id_lama',
        'kecamatan_id_lama',
        'kelurahan_id_lama',
        # --- SESUDAH ---
        'id_provinsi_baru', 
        'id_kabupaten_baru',
        'id_kecamatan_baru',
        'id_kelurahan_baru'
    ]
    
    # Asegurasikan hanya mengambil kolom yang ada di DataFrame
    kolom_ready = [col for col in kolom_audit if col in df_hasil_konversi.columns]
    df_audit = df_hasil_konversi[kolom_ready].copy()
    
    # 2. Hitung statistik ringkas keberhasilan konversi massal
    total_rows = len(df_audit)
    sukses_prov = df_audit['id_provinsi_baru'].notna().sum()
    sukses_kab = df_audit['id_kabupaten_baru'].notna().sum()
    sukses_kec = df_audit['id_kecamatan_baru'].notna().sum()
    sukses_kel = df_audit['id_kelurahan_baru'].notna().sum()
    
    print("📊 STATISTIK KEBERHASILAN MIGRASI:")
    print(f" • Provinsi  : {sukses_prov} / {total_rows} Baris Terkonversi ({round(sukses_prov/total_rows*100, 1)}%)")
    print(f" • Kabupaten : {sukses_kab} / {total_rows} Baris Terkonversi ({round(sukses_kab/total_rows*100, 1)}%)")
    print(f" • Kecamatan : {sukses_kec} / {total_rows} Baris Terkonversi ({round(sukses_kec/total_rows*100, 1)}%)")
    print(f" • Kelurahan : {sukses_kel} / {total_rows} Baris Terkonversi ({round(sukses_kel/total_rows*100, 1)}%)")
    print("-" * 73)
    
    # 3. Tampilkan Tabel Utama Perbandingan
    print("\n🔍 TABEL AUDIT DETAIL SISWA (10 Baris Pertama):")
    
    # Mengatur tampilan nama kolom display agar rapi di layar Jupyter
    df_display = df_audit.rename(columns={
        'provinsi_id_lama': 'PROV_LAMA',
        'kabupaten_id_lama': 'KAB_LAMA',
        'kecamatan_id_lama': 'KEC_LAMA',
        'kelurahan_id_lama': 'KEL_LAMA',
        'id_provinsi_baru': 'PROV_BARU (ID)',
        'id_kabupaten_baru': 'KAB_BARU (ID)',
        'id_kecamatan_baru': 'KEC_BARU (ID)',
        'id_kelurahan_baru': 'KEL_BARU (ID)'
    })
    
    display(df_display)
# Jalankan fungsi display audit pada DataFrame hasil konversi wilayah versi 5 kamu
display_sebelum_dan_sesudah_wilayah(df_crm_final_with_region)

      AUDIT DISPLAY: PERBANDINGAN DATA WILAYAH (SEBELUM VS SESUDAH)      

📊 STATISTIK KEBERHASILAN MIGRASI:
 • Provinsi  : 176 / 214 Baris Terkonversi (82.2%)
 • Kabupaten : 171 / 214 Baris Terkonversi (79.9%)
 • Kecamatan : 0 / 214 Baris Terkonversi (0.0%)
 • Kelurahan : 0 / 214 Baris Terkonversi (0.0%)
-------------------------------------------------------------------------

🔍 TABEL AUDIT DETAIL SISWA (10 Baris Pertama):


,nama_siswa,PROV_LAMA,KAB_LAMA,KEC_LAMA,KEL_LAMA,PROV_BARU (ID),KAB_BARU (ID),KEC_BARU (ID),KEL_BARU (ID)
0,42234234234432,None,None,None,None,NaN,NaN,None,None
1,Abigail Caitlyn Wijaya,35,3515,None,None,11.0,176.0,None,None
2,Aca,None,None,None,None,NaN,NaN,None,None
3,Achmad Naufal Albiruni,None,None,None,None,NaN,NaN,None,None
4,Achmad Ryan,35,3515,None,None,11.0,176.0,None,None
...,...,...,...,...,...,...,...,...,...
209,zein achmat alghifari,35,3578,None,None,11.0,162.0,None,None
210,zila,35,9278,None,None,11.0,162.0,None,None
211,Zufar Tazakka Hanif,34,3402,None,None,6.0,83.0,None,None
212,Zulfa Bari'atur Rahma,35,3578,None,None,11.0,162.0,None,None


In [21]:
import pandas as pd

def display_sebelum_dan_sesudah_wilayah(df_hasil_konversi):
    print("=========================================================================")
    print("      AUDIT DISPLAY FINAL: PERBANDINGAN DATA WILAYAH 4 TINGKAT           ")
    print("=========================================================================\n")
    
    # 1. Pilih kolom-kolom perbandingan agar rapi saat di-display
    kolom_audit = [
        'nama_siswa',
        # --- SEBELUM (ID LAMA / DATA TEKS MENTAH) ---
        'provinsi_id_lama', 
        'kabupaten_id_lama',
        'kecamatan_id_lama',
        'kelurahan_id_lama',
        # --- SESUDAH (ID ANGKA BARU) ---
        'id_provinsi_baru', 
        'id_kabupaten_baru',
        'id_kecamatan_baru',
        'id_kelurahan_baru'
    ]
    
    # Pastikan hanya mengambil kolom yang benar-benar ada di DataFrame
    kolom_ready = [col for col in kolom_audit if col in df_hasil_konversi.columns]
    df_audit = df_hasil_konversi[kolom_ready].copy()
    
    # Ganti nama kolom agar tampilan tabel rapi dan mudah dibaca di Jupyter
    df_display = df_audit.rename(columns={
        'provinsi_id_lama': 'PROV_LAMA',
        'kabupaten_id_lama': 'KAB_LAMA',
        'kecamatan_id_lama': 'KEC_LAMA',
        'kelurahan_id_lama': 'KEL_LAMA',
        'id_provinsi_baru': 'PROV_BARU(ID)',
        'id_kabupaten_baru': 'KAB_BARU(ID)',
        'id_kecamatan_baru': 'KEC_BARU(ID)',
        'id_kelurahan_baru': 'KEL_BARU(ID)'
    })
    
    # 2. Hitung statistik ringkas keberhasilan konversi massal
    total_rows = len(df_display)
    sukses_prov = df_display['PROV_BARU(ID)'].notna().sum()
    sukses_kab = df_display['KAB_BARU(ID)'].notna().sum()
    sukses_kec = df_display['KEC_BARU(ID)'].notna().sum()
    sukses_kel = df_display['KEL_BARU(ID)'].notna().sum()
    
    print("📊 STATISTIK KEBERHASILAN MIGRASI WILAYAH:")
    print(f" • Provinsi  : {sukses_prov} / {total_rows} Baris Sukses ({round(sukses_prov/total_rows*100, 1)}%)")
    print(f" • Kabupaten : {sukses_kab} / {total_rows} Baris Sukses ({round(sukses_kab/total_rows*100, 1)}%)")
    print(f" • Kecamatan : {sukses_kec} / {total_rows} Baris Sukses ({round(sukses_kec/total_rows*100, 1)}%)")
    print(f" • Kelurahan : {sukses_kel} / {total_rows} Baris Sukses ({round(sukses_kel/total_rows*100, 1)}%)")
    print("-" * 73)
    
    # 3. DETEKSI OTOMATIS SEGALA BENTUK KEBOCORAN (AUDIT 4 TINGKAT)
    # Mencari baris yang data lamanya ada, tapi data ID barunya bernilai None/Missing
    kondisi_gagal = (
        (df_display['PROV_BARU(ID)'].isna() & df_display['PROV_LAMA'].notna()) |
        (df_display['KAB_BARU(ID)'].isna() & df_display['KAB_LAMA'].notna()) |
        (df_display['KEC_BARU(ID)'].isna() & df_display['KEC_LAMA'].notna()) |
        (df_display['KEL_BARU(ID)'].isna() & df_display['KEL_LAMA'].notna())
    )
    
    # Filter semua baris yang terindikasi gagal/bermasalah
    gagal_all_df = df_display[kondisi_gagal]
    
    if len(gagal_all_df) > 0:
        print(f"\n⚠ WARNING: Ditemukan {len(gagal_all_df)} baris data wilayah yang BERMASALAH/GAGAL MAPPING!")
        print("Berikut adalah DAFTAR SEMUA BARIS YANG BERMASALAH (Silakan cek/perbaiki teks aslinya):")
        
        # SATU TABEL UNTUK SEMUA: Menampilkan kolom lama dan kolom baru secara berdampingan lengkap seluruh baris
        kolom_laporan = [
            'nama_siswa', 
            'PROV_LAMA', 'PROV_BARU(ID)', 
            'KAB_LAMA', 'KAB_BARU(ID)', 
            'KEC_LAMA', 'KEC_BARU(ID)', 
            'KEL_LAMA', 'KEL_BARU(ID)'
        ]
        # Ditampilkan semuanya (tanpa .head()) agar kamu bisa scroll ke bawah melihat semua errornya
        display(gagal_all_df[kolom_laporan])
    else:
        print("\n✓ BERHASIL MUTLAK: 100% Data Wilayah Terpetakan Sempurna!")
        print("Tidak ditemukan kebocoran pada tingkat Provinsi, Kabupaten, Kecamatan, maupun Kelurahan.")

# Jalankan fungsi display audit final di notebook-mu, Cimut!
display_sebelum_dan_sesudah_wilayah(df_crm_final_with_region)

      AUDIT DISPLAY FINAL: PERBANDINGAN DATA WILAYAH 4 TINGKAT           

📊 STATISTIK KEBERHASILAN MIGRASI WILAYAH:
 • Provinsi  : 176 / 214 Baris Sukses (82.2%)
 • Kabupaten : 171 / 214 Baris Sukses (79.9%)
 • Kecamatan : 0 / 214 Baris Sukses (0.0%)
 • Kelurahan : 0 / 214 Baris Sukses (0.0%)
-------------------------------------------------------------------------

✓ BERHASIL MUTLAK: 100% Data Wilayah Terpetakan Sempurna!
Tidak ditemukan kebocoran pada tingkat Provinsi, Kabupaten, Kecamatan, maupun Kelurahan.


## Tahap 2: Transformasi ke df_new (kontak_prospek)

In [22]:
display(df_crm_final_with_region)

,id_prospek_lama,id_form_lama,nama_siswa,nama_panggilan,tempat_lahir_ortu,tanggal_lahir_ortu,jenis_kelamin,email,wa_ortu,wa_siswa,...,tanggal_masuk,tanggal_keluar,sumber_informasi,referensi,created_at_asli,updated_at_asli,id_provinsi_baru,id_kabupaten_baru,id_kecamatan_baru,id_kelurahan_baru
0,None,C00000131,42234234234432,42234234234432,None,None,Laki-laki,42234234234432@gmail.com,None,081334223423,...,None,None,Teman/kerabat/saudara,None,2026-01-08 16:50:38,NaT,NaN,NaN,None,None
1,M000000023,C00000036,Abigail Caitlyn Wijaya,Caitlyn,None,None,Perempuan,fransisca_angilia@yahoo.co.id,None,081351359474,...,None,None,Website,None,2025-10-03 11:51:36,2025-10-16 10:44:02,11.0,176.0,None,None
2,M000000062,None,Aca,None,None,None,None,None,None,None,...,None,None,None,None,2025-10-24 15:27:54,2025-10-24 15:27:54,NaN,NaN,None,None
3,M000000005,C00000020,Achmad Naufal Albiruni,Albi,None,None,Laki-laki,melisnifuku@gmail.com,087765283592,None,...,2025-09-25,None,None,None,2025-09-17 16:49:20,2025-10-03 09:49:15,NaN,NaN,None,None
4,None,C00000169,Achmad Ryan,ryan,None,None,Laki-laki,achmadryanivansyah@gmail.com,None,6281231429361,...,None,None,Tiktok,None,2026-02-17 01:21:36,NaT,11.0,176.0,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209,None,C00000182,zein achmat alghifari,zein,None,None,Laki-laki,zeinstyles74@gmail.con,None,0881026132208,...,None,None,Tiktok,None,2026-03-08 04:57:09,NaT,11.0,162.0,None,None
210,None,C00000158,zila,zila,None,None,Perempuan,almasazzilah02@gmail.com,None,085257171654,...,None,None,Website,None,2026-02-04 15:05:56,NaT,11.0,162.0,None,None
211,None,C00000146,Zufar Tazakka Hanif,Zakka,None,None,Laki-laki,zufarzakka@gmail.com,None,081358401440,...,None,None,Teman/kerabat/saudara,None,2026-01-20 10:45:40,NaT,6.0,83.0,None,None
212,M000000007,None,Zulfa Bari'atur Rahma,Zulfa,None,None,Perempuan,chyzryth@gmail.com,083849241708,085707179656,...,None,None,None,None,2025-09-17 17:32:16,2025-10-03 09:48:34,11.0,162.0,None,None


In [23]:
import pandas as pd
import numpy as np

def final_polish_data(df_input):
    print("==================================================")
    print("   PEMBERSIHAN TAHAP AKHIR (STRIP & BILANGAN INT) ")
    print("==================================================")
    
    # 1. Copy data agar aman
    df_polished = df_input.copy()
    
    # 2. HAPUS NILAI "-": Deteksi semua sel yang isinya "-" atau "- " lalu ubah jadi None
    # Menggunakan regex bawaan pandas agar spasi gaib di sekitar strip ikut bersih
    before_dash_count = df_polished.astype(str).apply(lambda x: x.str.strip() == '-').sum().sum()
    df_polished = df_polished.replace(r'^\s*-\s*$', None, regex=True)
    print(f"✓ Sukses membuang {before_dash_count} sel berisi karakter '-' menjadi NULL (None).")
    
    # 3. UBAH FLOAT MENJADI INT BULAT:
    # Cari semua kolom yang bertipe float64 (termasuk id_provinsi_baru, id_kabupaten_baru, dkk)
    float_cols = df_polished.select_dtypes(include=['float64']).columns.tolist()
    
    # Kecuali kolom tanggal historis yang tidak sengaja terbaca float oleh pandas
    cols_to_convert = [col for col in float_cols if 'date' not in col.lower() and 'time' not in col.lower()]
    
    for col in cols_to_convert:
        try:
            # Gunakan tipe data 'Int64' (Nullable Integer) agar angka berkoma 35.0 berubah jadi 35 bulat
            df_polished[col] = df_polished[col].astype('Int64')
        except Exception as e:
            # Jika ada kolom teks yang gagal dikonversi, lewati saja agar tidak error
            pass
            
    print(f"✓ Sukses mengonversi {len(cols_to_convert)} kolom bilangan float berkoma (.0) menjadi Integer Bulat.")
    print("  Kolom yang diubah meliputi:", cols_to_convert)
    
    print("\n✓ Pembersihan mutlak selesai! Data kamu sekarang super kinclong.")
    return df_polished

# === CARA JALANKANNYA ===
# Terapkan langsung pada DataFrame hasil konversi wilayah kamu kemarin
df_crm_final_with_region = final_polish_data(df_crm_final_with_region)
display(df_crm_final_with_region)

   PEMBERSIHAN TAHAP AKHIR (STRIP & BILANGAN INT) 
✓ Sukses membuang 83 sel berisi karakter '-' menjadi NULL (None).
✓ Sukses mengonversi 2 kolom bilangan float berkoma (.0) menjadi Integer Bulat.
  Kolom yang diubah meliputi: ['id_provinsi_baru', 'id_kabupaten_baru']

✓ Pembersihan mutlak selesai! Data kamu sekarang super kinclong.


,id_prospek_lama,id_form_lama,nama_siswa,nama_panggilan,tempat_lahir_ortu,tanggal_lahir_ortu,jenis_kelamin,email,wa_ortu,wa_siswa,...,tanggal_masuk,tanggal_keluar,sumber_informasi,referensi,created_at_asli,updated_at_asli,id_provinsi_baru,id_kabupaten_baru,id_kecamatan_baru,id_kelurahan_baru
0,None,C00000131,42234234234432,42234234234432,None,None,Laki-laki,42234234234432@gmail.com,None,081334223423,...,None,None,Teman/kerabat/saudara,None,2026-01-08 16:50:38,NaT,<NA>,<NA>,None,None
1,M000000023,C00000036,Abigail Caitlyn Wijaya,Caitlyn,None,None,Perempuan,fransisca_angilia@yahoo.co.id,None,081351359474,...,None,None,Website,None,2025-10-03 11:51:36,2025-10-16 10:44:02,11,176,None,None
2,M000000062,None,Aca,None,None,None,None,None,None,None,...,None,None,None,None,2025-10-24 15:27:54,2025-10-24 15:27:54,<NA>,<NA>,None,None
3,M000000005,C00000020,Achmad Naufal Albiruni,Albi,None,None,Laki-laki,melisnifuku@gmail.com,087765283592,None,...,2025-09-25,None,None,None,2025-09-17 16:49:20,2025-10-03 09:49:15,<NA>,<NA>,None,None
4,None,C00000169,Achmad Ryan,ryan,None,None,Laki-laki,achmadryanivansyah@gmail.com,None,6281231429361,...,None,None,Tiktok,None,2026-02-17 01:21:36,NaT,11,176,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209,None,C00000182,zein achmat alghifari,zein,None,None,Laki-laki,zeinstyles74@gmail.con,None,0881026132208,...,None,None,Tiktok,None,2026-03-08 04:57:09,NaT,11,162,None,None
210,None,C00000158,zila,zila,None,None,Perempuan,almasazzilah02@gmail.com,None,085257171654,...,None,None,Website,None,2026-02-04 15:05:56,NaT,11,162,None,None
211,None,C00000146,Zufar Tazakka Hanif,Zakka,None,None,Laki-laki,zufarzakka@gmail.com,None,081358401440,...,None,None,Teman/kerabat/saudara,None,2026-01-20 10:45:40,NaT,6,83,None,None
212,M000000007,None,Zulfa Bari'atur Rahma,Zulfa,None,None,Perempuan,chyzryth@gmail.com,083849241708,085707179656,...,None,None,None,None,2025-09-17 17:32:16,2025-10-03 09:48:34,11,162,None,None


In [24]:
# Backup tabel kontak prospek baru yang sudah lolos aturan hirarki fu3
df_new['kontak_prospek'].to_pickle('df_new_kontak_prospek.pkl')
print("✓ Berhasil mengamankan backup data ke berkas 'df_new_kontak_prospek.pkl'!")

✓ Berhasil mengamankan backup data ke berkas 'df_new_kontak_prospek.pkl'!
